# LeetCode #192 — Word Frequency: đếm tần suất từ bằng ống lệnh Bash (`tr | sort | uniq -c | sort | awk`)

**Mức độ:** Medium · **Chủ đề:** Shell (Bash)

Đây là bài **Shell**: trên LeetCode bạn nộp **một dòng lệnh Bash** đọc file `words.txt`. Notebook chạy lệnh Bash thật (qua `subprocess`) và viết bản Python tương đương để hiểu từng bước.

Skeleton trên LeetCode:

```bash
# Read from the file words.txt and output the word frequency list to stdout.
```

Mục tiêu của notebook:

1. Hiểu **ống lệnh** (pipe `|`): đầu ra của lệnh trước là đầu vào của lệnh sau.
2. Bốn công cụ xử lý văn bản: `tr` (thay ký tự), `sort`, `uniq -c` (đếm dòng trùng **liền kề**), `awk` (xử lý theo cột).
3. Vì sao phải `sort` **trước** `uniq`; vì sao `sort -rn` (theo **số**, giảm dần).
4. Cách khác chỉ dùng `awk` (mảng kết hợp = dict). Bản Python với `Counter`.

# 1. Bài toán là gì?

Viết script Bash đếm tần suất mỗi từ trong file văn bản `words.txt`.

Giả thiết của đề:

- `words.txt` chỉ chứa chữ **thường** và dấu cách `' '`.
- Mỗi từ chỉ gồm chữ thường.
- Các từ cách nhau bởi **một hoặc nhiều** khoảng trắng.

In mỗi dòng `từ số_lần`, sắp theo **tần suất giảm dần**. Đề đảm bảo các tần suất **khác nhau** (không cần xử lý đồng hạng).

---

## Ví dụ

```text
words.txt:
the day is sunny the the
the sunny is is

Kết quả:
the 4
is 3
sunny 2
day 1
```

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Xử lý văn bản bằng Bash (pipe, tr, sort, uniq, awk)
2. Bảng băm (hash table) — `dict` và `set`

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Xử lý văn bản bằng Bash (pipe, tr, sort, uniq, awk)

### Triết lý Unix: nhiều lệnh nhỏ nối bằng **ống** `|`

Mỗi lệnh đọc văn bản từ **đầu vào chuẩn**, ghi ra **đầu ra chuẩn**; dấu `|` nối đầu ra lệnh trước vào đầu vào lệnh sau.

```text
cat words.txt | tr -s ' ' '\n' | sort | uniq -c | sort -rn
  đọc file      mỗi từ 1 dòng    xếp   đếm liền   xếp theo số giảm dần
```

| Lệnh | Tác dụng |
|---|---|
| `cat f` | in nội dung file |
| `tr -s ' ' '\n'` | thay ký tự (`-s`: gộp các ký tự lặp liên tiếp) |
| `sort` / `sort -rn` / `sort -k2` | sắp xếp (theo chữ / số giảm dần / theo cột 2) |
| `uniq -c` | gộp các dòng **giống nhau liền kề**, đếm số lần |
| `head -n 5`, `tail -n +2` | lấy 5 dòng đầu, bỏ dòng đầu |
| `awk '{...}'` | ngôn ngữ nhỏ xử lý từng dòng theo **trường** |

Trong notebook ta gọi Bash qua `subprocess`; nếu máy không có `bash` thì phần demo được bỏ qua (bản Python vẫn chạy).

In [1]:
import shutil, subprocess, os, tempfile

HAS_BASH_DEMO = shutil.which("bash") is not None

def bash_demo(cmd, files):
    if not HAS_BASH_DEMO:
        print("(máy không có bash — bỏ qua)")
        return
    with tempfile.TemporaryDirectory() as d:
        for name, content in files.items():
            with open(os.path.join(d, name), "w") as fh:
                fh.write(content)
        r = subprocess.run(["bash", "-c", cmd], cwd=d, capture_output=True, text=True)
        print(r.stdout, end="")

text = "the day is sunny the the\nthe sunny is is\n"
bash_demo("cat words.txt | tr -s ' ' '\\n' | sort | uniq -c | sort -rn", {"words.txt": text})

      4 the
      3 is
      2 sunny
      1 day


### `uniq` chỉ gộp các dòng **liền kề** → phải `sort` trước

In [2]:
bash_demo("printf 'a\\nb\\na\\n' | uniq -c", {})
print("---- sau khi sort:")
bash_demo("printf 'a\\nb\\na\\n' | sort | uniq -c", {})

      1 a
      1 b
      1 a
---- sau khi sort:
      2 a
      1 b


### awk: xử lý theo trường

- `$1, $2, ...` là các **trường** (tách theo khoảng trắng), `$0` là cả dòng, `NF` là số trường, `NR` là số thứ tự dòng.
- Khối `BEGIN {...}` chạy trước dòng đầu, `END {...}` chạy sau dòng cuối.
- Mảng của awk là **mảng kết hợp** (giống `dict`): `count[$1]++`.

In [3]:
bash_demo("awk '{ for (i = 1; i <= NF; i++) c[$i]++ } END { for (w in c) print w, c[w] }' words.txt | sort -k2 -rn",
          {"words.txt": text})
bash_demo("awk '{ print NR \": \" NF \" từ, từ cuối = \" $NF }' words.txt", {"words.txt": text})

the 4
is 3
sunny 2
day 1
1: 6 từ, từ cuối = the
2: 4 từ, từ cuối = is


### Tương đương bằng Python

In [4]:
from collections import Counter

cnt = Counter(text.split())
for w, c in sorted(cnt.items(), key=lambda kv: -kv[1]):
    print(w, c)

the 4
is 3
sunny 2
day 1


### Lỗi hay gặp

- `uniq -c` khi chưa `sort` → đếm sai.
- `sort` mặc định so sánh **chữ** (`"10" < "9"`) → dùng `-n` cho số.
- `tr ' ' '\n'` không có `-s` → nhiều khoảng trắng liên tiếp sinh **dòng rỗng**.
- Trích dẫn: lệnh awk đặt trong `'...'` để shell không thay `$1`.

## 2.2 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [5]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [6]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 374.04 ms | trên set: 0.076 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [7]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [8]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.3 Áp dụng vào bài này

Bài yêu cầu một ống lệnh Bash: tách mỗi từ thành một dòng (`tr -s ' ' '\n'`), **sắp xếp** để các từ giống nhau đứng cạnh nhau, `uniq -c` đếm, `sort -rn` theo số lần giảm dần, rồi `awk` đổi thứ tự cột. Bản Python tương đương là một **bảng băm đếm** (`Counter`).

# 3. Nền tảng: chạy lệnh Bash từ Python

Ta tạo file trong một thư mục tạm rồi chạy lệnh bằng `subprocess.run(["bash", "-c", lệnh])`. Trên Windows không có `bash` thì các ô Bash sẽ in thông báo bỏ qua; phần Python vẫn chạy được.

In [9]:
import os, random, shutil, subprocess, tempfile
from collections import Counter

HAS_BASH = shutil.which("bash") is not None


def run_bash(cmd: str, files: dict) -> str:
    """Tạo các file {tên: nội dung} trong thư mục tạm, chạy cmd bằng bash ở đó, trả stdout."""
    if not HAS_BASH:
        raise RuntimeError("máy này không có bash")
    with tempfile.TemporaryDirectory() as d:
        for name, content in files.items():
            with open(os.path.join(d, name), "w") as f:
                f.write(content)
        r = subprocess.run(["bash", "-c", cmd], cwd=d, capture_output=True, text=True)
        return r.stdout + (("[stderr] " + r.stderr) if r.stderr else "")


EXAMPLE = "the day is sunny the the\nthe sunny is is\n"
print("có bash:", HAS_BASH)
print(run_bash("cat words.txt", {"words.txt": EXAMPLE}))

có bash: True
the day is sunny the the
the sunny is is



## Ống lệnh `|`

`lệnh1 | lệnh2` nối **đầu ra chuẩn** (stdout) của `lệnh1` vào **đầu vào chuẩn** (stdin) của `lệnh2`. Mỗi công cụ làm một việc nhỏ; ghép lại thành xử lý phức tạp.

## Bốn công cụ cần biết

| Lệnh | Tác dụng | Ví dụ |
|---|---|---|
| `tr A B` | thay từng ký tự trong `A` bằng ký tự tương ứng trong `B` | `tr ' ' '\n'`: dấu cách → xuống dòng |
| `tr -s X` | **gộp** các ký tự `X` lặp liền nhau thành một | `tr -s ' '` |
| `sort` | sắp các **dòng** (mặc định theo chữ) | `sort -n` theo số, `-r` giảm dần, `-k2` theo cột 2 |
| `uniq -c` | gộp các dòng **trùng liền kề**, thêm số lần ở đầu | `3 is` |
| `awk '{print $2, $1}'` | tách mỗi dòng thành các cột `$1, $2, …` theo khoảng trắng, in theo ý | đổi chỗ hai cột |

# 4. Từng bước của ống lệnh

## Bước 1 — Mỗi từ một dòng: `tr -s ' ' '\n'`

In [10]:
step1 = "cat words.txt | tr -s ' ' '\\n'"
print(repr(run_bash(step1, {"words.txt": EXAMPLE})))

'the\nday\nis\nsunny\nthe\nthe\nthe\nsunny\nis\nis\n'


## Bước 2 — Sắp xếp để các từ giống nhau **đứng cạnh nhau**: `sort`

In [11]:
step2 = step1 + " | sort"
print(run_bash(step2, {"words.txt": EXAMPLE}))

day
is
is
is
sunny
sunny
the
the
the
the



## Bước 3 — Đếm các dòng trùng liền kề: `uniq -c`

In [12]:
step3 = step2 + " | uniq -c"
print(run_bash(step3, {"words.txt": EXAMPLE}))

      1 day
      3 is
      2 sunny
      4 the



## Bước 4 — Sắp theo số lần giảm dần: `sort -rn`

`-n` so sánh theo **giá trị số** (`10 > 9`), `-r` đảo chiều thành giảm dần.

In [13]:
step4 = step3 + " | sort -rn"
print(run_bash(step4, {"words.txt": EXAMPLE}))

      4 the
      3 is
      2 sunny
      1 day



## Bước 5 — Đổi thứ tự cột thành `từ số_lần`: `awk '{ print $2, $1 }'`

In [14]:
SOLUTION = "cat words.txt | tr -s ' ' '\\n' | sort | uniq -c | sort -rn | awk '{ print $2, $1 }'"
print(run_bash(SOLUTION, {"words.txt": EXAMPLE}))

the 4
is 3
sunny 2
day 1



# 5. Cách khác: chỉ dùng `awk` (mảng kết hợp)

`awk` có **mảng kết hợp** (giống `dict`): `c[$i]++` tăng bộ đếm của từ `$i`. Khối `END { … }` chạy sau khi đọc hết file. `NF` là số cột (số từ) của dòng hiện tại.

```bash
awk '{ for (i = 1; i <= NF; i++) c[$i]++ } END { for (w in c) print w, c[w] }' words.txt | sort -k2 -rn
```

`for (w in c)` duyệt **không theo thứ tự** → vẫn cần `sort -k2 -rn` (theo cột 2, số, giảm dần).

In [15]:
SOLUTION_AWK = "awk '{ for (i = 1; i <= NF; i++) c[$i]++ } END { for (w in c) print w, c[w] }' words.txt | sort -k2 -rn"
print(run_bash(SOLUTION_AWK, {"words.txt": EXAMPLE}))

the 4
is 3
sunny 2
day 1



Cách này còn xử lý được file có **nhiều dấu cách đầu dòng / dòng trống**, vì `awk` tự tách cột theo mọi khoảng trắng.

# 6. Bản Python tương đương (để hiểu và kiểm tra)

In [16]:
def word_frequency_py(text: str) -> str:
    cnt = Counter(text.split())                      # split() không tham số: tách theo mọi khoảng trắng
    lines = [f"{w} {c}" for w, c in sorted(cnt.items(), key=lambda kv: -kv[1])]
    return "\n".join(lines) + ("\n" if lines else "")

print(word_frequency_py(EXAMPLE))

the 4
is 3
sunny 2
day 1



Từng bước Python ứng với lệnh Bash: `split()` ≈ `tr -s ' ' '\n'`; `Counter` ≈ `sort | uniq -c`; `sorted(key=-count)` ≈ `sort -rn`; f-string ≈ `awk '{print $2, $1}'`.

# 7. Phiên bản nộp LeetCode

```bash
cat words.txt | tr -s ' ' '\n' | sort | uniq -c | sort -rn | awk '{ print $2, $1 }'
```

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<22} PASS")


def random_text():
    """Văn bản ngẫu nhiên có tần suất các từ đều khác nhau (đúng giả thiết của đề)."""
    words = random.sample(["an", "ba", "cu", "do", "em", "gi"], random.randint(1, 5))
    counts = random.sample(range(1, 9), len(words))
    bag = [w for w, c in zip(words, counts) for _ in range(c)]
    random.shuffle(bag)
    out, line = [], []
    for w in bag:
        line.append(w)
        if random.random() < 0.25:
            out.append((" " * random.randint(1, 3)).join(line)); line = []
    if line:
        out.append(" ".join(line))
    return "\n".join(out) + "\n"


check("Python, ví dụ", word_frequency_py(EXAMPLE), "the 4\nis 3\nsunny 2\nday 1\n")
if HAS_BASH:
    check("Bash ống lệnh, ví dụ", run_bash(SOLUTION, {"words.txt": EXAMPLE}), "the 4\nis 3\nsunny 2\nday 1\n")
    check("Bash awk, ví dụ", run_bash(SOLUTION_AWK, {"words.txt": EXAMPLE}), "the 4\nis 3\nsunny 2\nday 1\n")
    random.seed(192)
    for _ in range(100):
        txt = random_text()
        exp = word_frequency_py(txt)
        assert run_bash(SOLUTION, {"words.txt": txt}) == exp == run_bash(SOLUTION_AWK, {"words.txt": txt}), txt
    print("100 file ngẫu nhiên (nhiều dấu cách, nhiều dòng): hai lệnh Bash khớp bản Python")
else:
    print("(bỏ qua phần Bash vì không có bash)")

Python, ví dụ          PASS


Bash ống lệnh, ví dụ   PASS
Bash awk, ví dụ        PASS


100 file ngẫu nhiên (nhiều dấu cách, nhiều dòng): hai lệnh Bash khớp bản Python


# 8. Vì sao đúng? Độ phức tạp?

- Sau `tr -s ' ' '\n'`, mỗi từ là một dòng (các dấu cách liền nhau gộp thành một xuống dòng).
- `sort` đặt các dòng giống nhau **liền kề**; `uniq -c` gộp **mỗi khối liền kề** thành một dòng kèm số lần → đếm đúng tần suất.
- `sort -rn` sắp theo số ở đầu dòng, giảm dần; tần suất đều khác nhau nên thứ tự là duy nhất.

```text
Thời gian: O(N log N) với N = số từ (do sort)
Bộ nhớ:    O(N) cho sort (bản awk: O(số từ khác nhau) + sort kết quả)
```

# 9. Những lỗi dễ mắc

## Lỗi 1: `uniq -c` mà không `sort` trước

In [18]:
if HAS_BASH:
    print(run_bash("cat words.txt | tr -s ' ' '\\n' | uniq -c", {"words.txt": EXAMPLE}))

      1 the
      1 day
      1 is
      1 sunny
      3 the
      1 sunny
      2 is



`uniq` chỉ gộp các dòng trùng **liền kề**: `the` xuất hiện ở nhiều chỗ rời nhau nên bị đếm thành nhiều dòng `1 the`, `2 the`, ….

## Lỗi 2: `sort -r` (theo chữ) thay vì `sort -rn` (theo số)

In [19]:
BIG = ("a " * 10 + "b " * 9 + "c ").strip() + "\n"
AWK_COUNT = "awk '{ for (i = 1; i <= NF; i++) c[$i]++ } END { for (w in c) print w, c[w] }' words.txt"
if HAS_BASH:
    print("ống lệnh uniq, sort -r :", run_bash("cat words.txt | tr -s ' ' '\\n' | sort | uniq -c | sort -r | awk '{print $2, $1}'", {"words.txt": BIG}).split("\n")[:3])
    print("bản awk,  sort -k2 -r  :", run_bash(AWK_COUNT + " | sort -k2 -r", {"words.txt": BIG}).split("\n")[:3])
    print("bản awk,  sort -k2 -rn :", run_bash(AWK_COUNT + " | sort -k2 -rn", {"words.txt": BIG}).split("\n")[:3])

ống lệnh uniq, sort -r : ['a 10', 'b 9', 'c 1']
bản awk,  sort -k2 -r  : ['b 9', 'a 10', 'c 1']
bản awk,  sort -k2 -rn : ['a 10', 'b 9', 'c 1']


- Ngay sau `uniq -c`, `sort -r` **tình cờ** đúng: `uniq -c` căn phải số đếm bằng khoảng trắng (`"     10 a"`, `"      9 b"`), nên so theo chữ cũng ra đúng thứ tự số (cho tới khi số đếm dài hơn độ rộng đệm).
- Với bản `awk` (không đệm), so theo chữ cho `"9" > "10"` (vì `'9' > '1'`) → `b 9` đứng trước `a 10` — **sai**.

Luôn dùng `-n` khi sắp theo số, đừng dựa vào định dạng đệm.

## Lỗi 3: `tr ' ' '\n'` không có `-s` khi có nhiều dấu cách liền nhau

In [20]:
SPACY = "a  b\na   a\n"
if HAS_BASH:
    print("không -s:", repr(run_bash("cat words.txt | tr ' ' '\\n' | sort | uniq -c | sort -rn | awk '{print $2, $1}'", {"words.txt": SPACY})))
    print("có -s   :", repr(run_bash(SOLUTION, {"words.txt": SPACY})))

không -s: 'a 3\n 3\nb 1\n'


có -s   : 'a 3\nb 1\n'


Hai dấu cách liền nhau tạo ra **dòng rỗng**; `uniq -c` đếm cả dòng rỗng, `awk` in ra một dòng chỉ có số (từ rỗng). Bản awk-only (mục 5) không bị vì `awk` bỏ qua khoảng trắng thừa.

## Lỗi 4: Quên đổi chỗ cột

`uniq -c` in `số từ`; đề yêu cầu `từ số`. Thiếu bước `awk '{print $2, $1}'` sẽ bị chấm sai định dạng.

# 10. Mẹo nhận diện pattern

- **Đếm tần suất trong Shell** → `sort | uniq -c | sort -rn`, hoặc `awk` với mảng kết hợp.
- **Tách / gộp ký tự** → `tr` (`-s` gộp, `-d` xóa).
- **Theo cột** → `awk '{print $k}'`, `cut -d' ' -fk`, `sort -k`.

Bài liên quan: **#193 Valid Phone Numbers** (Easy, `grep -E`), **#194 Transpose File**, **#195 Tenth Line** (Easy, `sed -n '10p'`), **#347 Top K Frequent Elements** (cùng ý tưởng bằng Python).

# 11. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Với `words.txt` = `"b a c a\nb a\n"`, ghi đầu ra sau mỗi bước của ống lệnh.

## Bài 2 — Bản Python

Viết hàm trả về đúng chuỗi mà lệnh Bash in ra.

In [21]:
def exercise_2_word_frequency(text: str) -> str:
    # TODO
    pass


# assert exercise_2_word_frequency("the day is sunny the the\nthe sunny is is\n") == "the 4\nis 3\nsunny 2\nday 1\n"
# assert exercise_2_word_frequency("  x  y y \n") == "y 2\nx 1\n"

## Bài 3 — Bash: chỉ in **2 từ** xuất hiện nhiều nhất

Trả về chuỗi lệnh Bash (đọc `words.txt`). Gợi ý: `head -n 2`.

In [22]:
def exercise_3_top2_cmd() -> str:
    # TODO
    pass


# assert (not HAS_BASH) or run_bash(exercise_3_top2_cmd(), {"words.txt": "the day is sunny the the\nthe sunny is is\n"}) == "the 4\nis 3\n"

## Bài 4 — Biến thể Python: có **đồng hạng**, sắp theo tần suất giảm dần rồi theo chữ cái

In [23]:
def exercise_4_frequency_with_ties(text: str) -> list:
    # TODO: trả về list (từ, số_lần)
    pass


# assert exercise_4_frequency_with_ties("b a c a b\n") == [("a", 2), ("b", 2), ("c", 1)]
# assert exercise_4_frequency_with_ties("") == []

## Bài 5 — Mini challenge: Bash xử lý đồng hạng như bài 4

Sắp theo cột 2 giảm dần theo số, đồng hạng thì cột 1 tăng dần theo chữ: `sort -k2,2nr -k1,1`.

In [24]:
def exercise_5_ties_cmd() -> str:
    # TODO
    pass


# assert (not HAS_BASH) or run_bash(exercise_5_ties_cmd(), {"words.txt": "b a c a b\n"}) == "a 2\nb 2\nc 1\n"

# 12. Gợi ý

**Bài 1**
- Mức 1: chạy `run_bash(step1/step2/step3/step4, ...)` với file mới.

**Bài 2**
- Mức 1: `Counter(text.split())`, sắp theo `-count`, ghép `"từ số"` từng dòng, mỗi dòng kết thúc bằng `"\n"`.

**Bài 4**
- Mức 1: `sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))`.

**Bài 5**
- Mức 1: dùng bản awk-only để có `từ số`, rồi `sort -k2,2nr -k1,1`.
- Mức 2: `-k2,2nr` nghĩa là "khóa là đúng cột 2, so theo số, giảm dần"; `-k1,1` là khóa phụ.

# 13. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [25]:
f1 = {"words.txt": "b a c a\nb a\n"}
if HAS_BASH:
    for name, cmd in [("tr", step1), ("sort", step2), ("uniq -c", step3), ("sort -rn", step4), ("awk", SOLUTION)]:
        print(f"--- sau {name}:")
        print(run_bash(cmd, f1), end="")

--- sau tr:


b
a
c
a
b
a
--- sau sort:
a
a
a
b
b
c
--- sau uniq -c:
      3 a
      2 b
      1 c
--- sau sort -rn:
      3 a
      2 b
      1 c
--- sau awk:


a 3
b 2
c 1


## Đáp án Bài 2

In [26]:
def answer_2_word_frequency(text: str) -> str:
    return word_frequency_py(text)

assert answer_2_word_frequency("the day is sunny the the\nthe sunny is is\n") == "the 4\nis 3\nsunny 2\nday 1\n"
assert answer_2_word_frequency("  x  y y \n") == "y 2\nx 1\n"
print("Bài 2 PASS (giống word_frequency_py ở mục 6)")

Bài 2 PASS (giống word_frequency_py ở mục 6)


## Đáp án Bài 3

In [27]:
def answer_3_top2_cmd() -> str:
    return SOLUTION + " | head -n 2"

assert (not HAS_BASH) or run_bash(answer_3_top2_cmd(), {"words.txt": "the day is sunny the the\nthe sunny is is\n"}) == "the 4\nis 3\n"
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [28]:
def answer_4_frequency_with_ties(text: str) -> list:
    cnt = Counter(text.split())
    return sorted(cnt.items(), key=lambda kv: (-kv[1], kv[0]))

assert answer_4_frequency_with_ties("b a c a b\n") == [("a", 2), ("b", 2), ("c", 1)]
assert answer_4_frequency_with_ties("") == []
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [29]:
def answer_5_ties_cmd() -> str:
    return "awk '{ for (i = 1; i <= NF; i++) c[$i]++ } END { for (w in c) print w, c[w] }' words.txt | sort -k2,2nr -k1,1"

assert (not HAS_BASH) or run_bash(answer_5_ties_cmd(), {"words.txt": "b a c a b\n"}) == "a 2\nb 2\nc 1\n"
if HAS_BASH:
    for _ in range(100):
        words = [random.choice("abcde") for _ in range(random.randint(1, 15))]
        txt = " ".join(words) + "\n"
        exp = "".join(f"{w} {c}\n" for w, c in answer_4_frequency_with_ties(txt))
        assert run_bash(answer_5_ties_cmd(), {"words.txt": txt}) == exp, txt
print("Bài 5 PASS")

Bài 5 PASS


# 14. Tổng kết

> `cat words.txt | tr -s ' ' '\n' | sort | uniq -c | sort -rn | awk '{ print $2, $1 }'`: tách mỗi từ một dòng, sắp để từ trùng liền kề, `uniq -c` đếm, `sort -rn` sắp theo số giảm dần, `awk` đổi chỗ cột. Hoặc đếm bằng mảng kết hợp của `awk` rồi `sort -k2 -rn`.

---

## Checklist trước khi nộp

- [ ] `sort` trước `uniq -c`?
- [ ] `sort -rn` (có `-n`)?
- [ ] `tr -s` để gộp dấu cách liền nhau?
- [ ] Định dạng `từ số`?